# Детализация всплеска СФП (2025-02)

Тетрадка для расшифровки всплеска СФП по коду фактора и наименованию.

По умолчанию анализирует:
- сегмент: `МСБ`
- месяц: `2025-02`

In [ ]:
import warnings
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", None)

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"
REF_FILE = f"{DATA_DIR}/ref_book.csv"

DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO = pd.Timestamp("2025-12-31")
ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}

# Параметры детализации (можно менять)
TARGET_SEGMENT = "МСБ"
TARGET_MONTH = "2025-02"


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


In [ ]:
# 1) Загружаем и подготавливаем CRM в той же логике, что основной отчет
df = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
df.columns = df.columns.str.strip()

df["inn"] = df["X_INN"].apply(normalize_inn)
df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()

if "VAL" in df.columns:
    df = df[df["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()

df["TYPE_FP"] = df["TYPE_FP"].str.strip().replace({"FP": "ФП", "SFP": "СФП"})
df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)
df = df[df["segment"].notna()].copy()

df["fp_num"] = df["NUMBER_FP_SFP"].astype(str).str.strip()
df = df.dropna(subset=["inn", "NUMBER_FP_SFP"]).copy()
df = df.drop_duplicates(subset=["inn", "fp_num", "TYPE_FP", "IDENTIFICATION_DATE"]).copy()
df["year_month"] = df["dt"].dt.to_period("M").astype(str)

print(f"Готово: {len(df):,} строк")
print(f"Уникальных ИНН: {df['inn'].nunique():,}")

In [ ]:
# 2) Отбираем СФП за целевой месяц и сегмент
sfp_spike = df[
    (df["TYPE_FP"] == "СФП") &
    (df["segment"] == TARGET_SEGMENT) &
    (df["year_month"] == TARGET_MONTH)
].copy()

n_rows = len(sfp_spike)
n_inn = sfp_spike["inn"].nunique()

print("=" * 72)
print(f"СФП детализация: сегмент={TARGET_SEGMENT}, месяц={TARGET_MONTH}")
print("=" * 72)
print(f"Всего событий СФП: {n_rows:,}")
print(f"Уникальных ИНН:    {n_inn:,}")

In [ ]:
# 3) Детализация: какие именно СФП сформировали всплеск
detail = (
    sfp_spike
    .groupby("NUMBER_FP_SFP", as_index=False)
    .agg(
        Количество=("inn", "size"),
        Уникальных_ИНН=("inn", "nunique"),
    )
    .sort_values("Количество", ascending=False)
)

if len(detail) == 0:
    print("Нет данных по заданному месяцу/сегменту")
else:
    detail["Доля_событий_%"] = (detail["Количество"] / detail["Количество"].sum() * 100).round(2)

    # Подтягиваем наименования факторов из справочника
    ref = pd.read_csv(REF_FILE, sep="\t", encoding="utf-16", dtype=str)
    ref.columns = ref.columns.str.strip()
    ref_name = (
        ref[["№", "Наименование"]]
        .dropna(subset=["№"])
        .drop_duplicates(subset=["№"])
        .rename(columns={"№": "NUMBER_FP_SFP", "Наименование": "Наименование"})
    )

    detail = detail.merge(ref_name, on="NUMBER_FP_SFP", how="left")
    detail = detail[["NUMBER_FP_SFP", "Наименование", "Количество", "Доля_событий_%", "Уникальных_ИНН"]]
    detail = detail.reset_index(drop=True)
    detail.index = detail.index + 1

    display(detail)


In [ ]:
# 4) Визуализация TOP-10 СФП, сформировавших всплеск
if 'detail' in globals() and len(detail) > 0:
    top = detail.head(10).copy()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(top["NUMBER_FP_SFP"].astype(str), top["Количество"], color="#4C78A8", edgecolor="white")
    ax.set_title(f"TOP-10 СФП в {TARGET_SEGMENT}, {TARGET_MONTH}", fontsize=13, fontweight="bold")
    ax.set_xlabel("№ СФП")
    ax.set_ylabel("Количество событий")
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    # Опционально: сохраняем детализацию в Excel для слайда
    out_path = f"{DATA_DIR}/SFP_spike_{TARGET_SEGMENT}_{TARGET_MONTH}.xlsx"
    detail.to_excel(out_path, index=False)
    print(f"Сохранено: {out_path}")
